In [1]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map=device,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /Users/hervind/hvd-code/visual_language/venv/lib/python3.12/site-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /Users/hervind/hvd-code/visual_language/venv/lib/python3.12/site-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


In [2]:
def describe_image(image_url: str, prompt: str, max_new_tokens: int = 256) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_url},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return output_text

In [3]:
image_url = "https://storage.ghost.io/c/db/e7/dbe79357-2349-45c5-9230-4be384c8629b/content/images/2019/12/invoice-sample.jpg"
prompt = "Describe this image, what is the valuable information from this"

output_text = describe_image(
    image_url,
    prompt
)
print(output_text)

This image is a **professional invoice template** from a company named “Your Company LLC.” It is designed to be clear, organized, and visually appealing, with a modern layout featuring a blue and yellow logo on the top left and a stylized icon on the top right.

---

### **Key Valuable Information:**

#### **1. Invoice Header & Company Info**
- **Invoice Title**: “Invoice” in bold, large font.
- **Company Name**: “Your Company LLC”
- **Company Address**: “Address 123, State, My Country”
- **Contact Info**: Phone: 111-222-333, Fax: 111-222-334
- **Website**: `http://mrsinvoice.com`

#### **2. Billing & Shipping Details**
- **Bill To** (Client):
  - Name: John Doe
  - Address: Alpha Bravo Road 33
  - Phone: 111-222-333ант, Fax: 111-222-334
  - Email: `client@example.net`

- **Shipping To** (Office):
  - Name: John Doe Office
 
